# ECSC: Within-Child Signature Stability Analysis

## Research Question
Do children have stable "memory-use profiles" (shape metrics) that persist across sessions?

## Design
- ECSC has **longitudinal data**: 70 children with 3 sessions (YR1, YR2, YR3) spanning ages 5-11
- Challenge: Tasks (frog stories) are **counterbalanced** across years
- Approach: First test task ICC, then test child-level ICC

## Metrics
- **Shape metrics** (primary): `log_slope_local`, `auc_log_k`, `auc_linear_k`, `early_ratio`
- **Level metrics** (secondary): `mean_gain_128`, `baseline_nll`

## Analysis Steps
1. Load ECSC transcripts with metadata
2. Run local lag curve analysis (LLM-based)
3. Compute task ICC (is metric task-dependent?)
4. Compute child-level ICC (is metric child-dependent?)
5. Compute within-child vs between-child distances
6. Test developmental trends across age groups

## Interpretation
- If child ICC is meaningful (~0.15–0.30) and within/between ratio < 1: **signature exists**
- If child ICC near zero and within/between ~1: metric is local/contextual only

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths - UPDATE THESE FOR YOUR DRIVE
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/ecsc"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/ECSC"
TRANSCRIPTS_PATH = f"{DRIVE_BASE}/transcripts.jsonl"
INVENTORY_PATH = f"{DRIVE_BASE}/ecsc_file_inventory_full.csv"

EXPERIMENT = 'within_child_signature_v1'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Sliding window parameters
TARGET_LENGTH = 32  # T=32 tokens per window
N_WINDOWS = 10      # Fewer windows (smaller texts)
MIN_CONTEXT = 64    # Min tokens of preceding context (smaller for child speech)
END_BUFFER = 32     # Exclude last 32 tokens

# Context length grid (k values)
K_GRID = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]

# Minimum tokens required (lower for child narratives)
MIN_TOKENS_REQUIRED = 100

# Reproducibility
RANDOM_SEED = 42
N_BOOTSTRAP = 100

# Create output directory
output_dir = Path(f"{OUTPUT_BASE}/{EXPERIMENT}")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT}")
print(f"Transcripts: {TRANSCRIPTS_PATH}")
print(f"Inventory: {INVENTORY_PATH}")
print(f"Output to: {output_dir}")
print(f"\nWindow parameters:")
print(f"  Target length T: {TARGET_LENGTH} tokens")
print(f"  Windows per transcript: {N_WINDOWS}")
print(f"  Min preceding context: {MIN_CONTEXT} tokens")
print(f"  Min tokens required: {MIN_TOKENS_REQUIRED}")

## 1. Load Data

In [ ]:
# Load transcripts
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

transcripts = load_jsonl(TRANSCRIPTS_PATH)
print(f"Loaded {len(transcripts)} transcripts")

# Parse population metadata
for t in transcripts:
    pop = json.loads(t['population'])
    t['subject_id'] = pop.get('subject_id', t['author_id'])
    t['age_months'] = pop.get('age_months')
    t['sex'] = pop.get('sex')
    t['study_year'] = pop.get('study_year')

df_transcripts = pd.DataFrame(transcripts)
print(f"\nUnique subjects: {df_transcripts['subject_id'].nunique()}")
print(f"Study years: {df_transcripts['study_year'].value_counts().to_dict()}")

In [ ]:
# Load inventory for task information
try:
    df_inventory = pd.read_csv(INVENTORY_PATH)
    print(f"Loaded inventory: {len(df_inventory)} rows")
    print(f"Task identified in: {df_inventory['task'].notna().sum()} / {len(df_inventory)} files")
    
    # Create lookup key from subject_id and yr
    df_inventory['lookup_key'] = df_inventory['subject_id'].astype(str) + '_YR' + df_inventory['yr'].str.replace('YR', '')
    task_lookup = df_inventory.set_index('lookup_key')['task'].to_dict()
    
    print(f"\nTask distribution:")
    print(df_inventory['task'].value_counts(dropna=False))
    
except FileNotFoundError:
    print("Inventory file not found. Will proceed without task information.")
    task_lookup = {}

In [ ]:
# Add task to transcripts
for t in transcripts:
    key = f"{t['subject_id']}_YR{t['study_year']}"
    t['task'] = task_lookup.get(key)

df_transcripts = pd.DataFrame(transcripts)

# Summary
print("\nTranscript summary:")
print(f"  Total: {len(df_transcripts)}")
print(f"  With task: {df_transcripts['task'].notna().sum()}")
print(f"  Unique subjects: {df_transcripts['subject_id'].nunique()}")

# Longitudinal structure
sessions_per_child = df_transcripts.groupby('subject_id').size()
print(f"\nSessions per child:")
print(sessions_per_child.value_counts().sort_index())

# Children with 2+ sessions (for ICC)
multi_session_children = sessions_per_child[sessions_per_child >= 2].index.tolist()
print(f"\nChildren with 2+ sessions: {len(multi_session_children)}")

In [ ]:
# Age groups for developmental analysis
def age_group(age_months):
    if age_months is None:
        return 'unknown'
    if age_months < 72:
        return '5yr'
    elif age_months < 84:
        return '6yr'
    elif age_months < 96:
        return '7yr'
    elif age_months < 108:
        return '8yr'
    elif age_months < 120:
        return '9yr'
    else:
        return '10+yr'

df_transcripts['age_group'] = df_transcripts['age_months'].apply(age_group)
print("Age group distribution:")
print(df_transcripts['age_group'].value_counts().sort_index())

## 2. Load Model

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Tokenize and filter
transcript_tokens = {}
excluded_ids = []

for t in transcripts:
    token_ids = tokenizer.encode(t['text'], add_special_tokens=False)
    if len(token_ids) >= MIN_TOKENS_REQUIRED:
        transcript_tokens[t['doc_id']] = token_ids
    else:
        excluded_ids.append(t['doc_id'])

print(f"Tokenized transcripts: {len(transcript_tokens)}")
print(f"Excluded (too short): {len(excluded_ids)}")

lengths = [len(t) for t in transcript_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

## 3. Core Functions

In [ ]:
@torch.no_grad()
def compute_nll_for_target(context_ids, target_ids):
    """Compute mean NLL for predicting target_ids given context_ids."""
    full_ids = context_ids + target_ids
    
    if len(context_ids) == 0:
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        nlls = []
        for i in range(len(full_ids) - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            true_token = full_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0
    else:
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        target_start = len(context_ids)
        nlls = []
        
        log_probs = torch.log_softmax(logits[target_start - 1], dim=-1)
        nll_first = -log_probs[target_ids[0]].item()
        nlls.append(nll_first)
        
        for i in range(len(target_ids) - 1):
            pos = target_start + i
            if pos >= logits.shape[0]:
                break
            log_probs = torch.log_softmax(logits[pos], dim=-1)
            true_token = target_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0


def sample_window_positions(n_tokens, n_windows, rng):
    """Sample target window start positions."""
    min_start = MIN_CONTEXT
    max_start = n_tokens - TARGET_LENGTH - END_BUFFER
    
    if max_start <= min_start:
        # Text too short for multiple windows
        if n_tokens >= MIN_CONTEXT + TARGET_LENGTH:
            return [MIN_CONTEXT]
        return []
    
    if n_windows == 1:
        return [(min_start + max_start) // 2]
    
    positions = np.linspace(min_start, max_start, n_windows, dtype=int).tolist()
    
    jittered = []
    for pos in positions:
        jitter = rng.randint(-3, 3)
        new_pos = max(min_start, min(max_start, pos + jitter))
        jittered.append(new_pos)
    
    return jittered


def run_lag_curve_for_window(token_ids, target_start):
    """Compute NLL and gain for each k for a single target window."""
    target_end = target_start + TARGET_LENGTH
    target_ids = token_ids[target_start:target_end]
    context_pool = token_ids[:target_start]
    
    nll_by_k = {}
    
    for k in K_GRID:
        if k == 0:
            context_ids = []
        else:
            context_ids = context_pool[-k:] if k <= len(context_pool) else context_pool
        
        nll = compute_nll_for_target(context_ids, target_ids)
        nll_by_k[k] = nll
    
    nll_0 = nll_by_k[0]
    gain_by_k = {k: nll_0 - nll_by_k[k] for k in K_GRID}
    
    return {'nll': nll_by_k, 'gain': gain_by_k}


def compute_auc_log_k(gains_by_k):
    """Compute AUC on log scale (emphasizes early saturation)."""
    auc = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        g1, g2 = gains_by_k[k1], gains_by_k[k2]
        width = np.log(k2 + 1) - np.log(k1 + 1)
        auc += 0.5 * (g1 + g2) * width
    return auc


def compute_auc_linear_k(gains_by_k):
    """Compute AUC on linear scale (emphasizes longer-range)."""
    auc = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        g1, g2 = gains_by_k[k1], gains_by_k[k2]
        width = k2 - k1
        auc += 0.5 * (g1 + g2) * width
    return auc


def compute_shape_metrics_from_gains(mean_gains):
    """Compute shape metrics from gains by k."""
    results = {}
    
    # early_ratio
    if mean_gains[128] > 0:
        results['early_ratio'] = mean_gains[16] / mean_gains[128]
    else:
        results['early_ratio'] = np.nan
    
    # log_slope_local
    ks_for_fit = [k for k in K_GRID if k >= 2]
    log_ks = [np.log(k + 1) for k in ks_for_fit]
    gains_for_fit = [mean_gains[k] for k in ks_for_fit]
    
    if len(log_ks) >= 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(log_ks, gains_for_fit)
        results['log_slope_local'] = slope
        results['log_slope_r2'] = r_value ** 2
    else:
        results['log_slope_local'] = np.nan
        results['log_slope_r2'] = np.nan
    
    # AUC metrics
    results['auc_log_k'] = compute_auc_log_k(mean_gains)
    results['auc_linear_k'] = compute_auc_linear_k(mean_gains)
    
    return results


def run_analysis_for_transcript(token_ids, doc_id, rng):
    """Run sliding window lag curve analysis for a single transcript."""
    n_tokens = len(token_ids)
    positions = sample_window_positions(n_tokens, N_WINDOWS, rng)
    
    if len(positions) == 0:
        return None
    
    all_gains = {k: [] for k in K_GRID}
    all_nlls = {k: [] for k in K_GRID}
    
    for target_start in positions:
        win_result = run_lag_curve_for_window(token_ids, target_start)
        
        for k in K_GRID:
            all_gains[k].append(win_result['gain'][k])
            all_nlls[k].append(win_result['nll'][k])
    
    # Aggregate
    results = {'doc_id': doc_id, 'n_windows': len(positions), 'token_count': n_tokens}
    
    mean_gains = {}
    for k in K_GRID:
        results[f'mean_nll_{k}'] = np.mean(all_nlls[k])
        results[f'mean_gain_{k}'] = np.mean(all_gains[k])
        mean_gains[k] = results[f'mean_gain_{k}']
    
    # Shape metrics
    shape_metrics = compute_shape_metrics_from_gains(mean_gains)
    results.update(shape_metrics)
    
    # Baseline NLL
    results['baseline_nll'] = results['mean_nll_128']
    
    return results


print("Core functions defined")

## 4. Run Analysis

In [ ]:
# Build transcript metadata lookup
transcript_meta = {t['doc_id']: t for t in transcripts}

# Initialize RNG
rng = random.Random(RANDOM_SEED)

# Process all transcripts
all_results = []
start_time = time.time()

for doc_id, token_ids in tqdm(transcript_tokens.items(), desc="Processing transcripts"):
    result = run_analysis_for_transcript(token_ids, doc_id, rng)
    
    if result is None:
        continue
    
    # Add metadata
    meta = transcript_meta[doc_id]
    result['subject_id'] = meta['subject_id']
    result['age_months'] = meta['age_months']
    result['age_group'] = age_group(meta['age_months'])
    result['sex'] = meta['sex']
    result['study_year'] = meta['study_year']
    result['task'] = meta.get('task')
    
    all_results.append(result)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(transcript_tokens):.2f}s/transcript)")
print(f"Transcripts processed: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results: {df.shape}")
print(f"\nUnique subjects: {df['subject_id'].nunique()}")
print(f"With task identified: {df['task'].notna().sum()} / {len(df)}")

# Sessions per subject
sessions = df.groupby('subject_id').size()
print(f"\nSessions per subject:")
print(sessions.value_counts().sort_index())

df.head()

## 5. Task ICC Analysis

**Question**: How much of the variance in shape metrics is attributable to task (story)?
- If ICC(task) is near zero: task doesn't matter, proceed with longitudinal
- If ICC(task) is substantial: need to control for task

In [ ]:
def compute_icc_oneway(df, metric_col, group_col):
    """Compute ICC(1,1) using one-way random effects ANOVA."""
    data = df[[group_col, metric_col]].dropna()
    
    if data[group_col].nunique() < 2:
        return None
    
    groups = data.groupby(group_col)[metric_col]
    n_groups = groups.ngroups
    group_means = groups.mean()
    group_sizes = groups.size()
    grand_mean = data[metric_col].mean()
    
    ss_between = sum(n * (m - grand_mean)**2 for n, m in zip(group_sizes, group_means))
    df_between = n_groups - 1
    
    ss_within = sum(((g - g.mean())**2).sum() for _, g in groups)
    df_within = len(data) - n_groups
    
    ms_between = ss_between / df_between if df_between > 0 else 0
    ms_within = ss_within / df_within if df_within > 0 else 0
    
    k = len(data) / n_groups
    
    if ms_between + (k - 1) * ms_within > 0:
        icc = (ms_between - ms_within) / (ms_between + (k - 1) * ms_within)
    else:
        icc = 0
    
    var_between = max(0, (ms_between - ms_within) / k)
    var_within = ms_within
    var_total = var_between + var_within
    
    return {
        'icc': icc,
        'var_between': var_between,
        'var_within': var_within,
        'var_total': var_total,
        'pct_between': 100 * var_between / var_total if var_total > 0 else 0,
        'n_groups': n_groups,
        'n_obs': len(data),
        'k_avg': k,
    }

In [ ]:
print("="*80)
print("TASK ICC ANALYSIS: Does story explain variance in shape metrics?")
print("="*80)

# Filter to transcripts with task identified
df_with_task = df[df['task'].notna()].copy()
print(f"\nTranscripts with task: {len(df_with_task)}")
print(f"Unique tasks: {df_with_task['task'].nunique()}")
print(f"\nTask distribution:")
print(df_with_task['task'].value_counts())

# Compute task ICC
task_icc_results = {}
metrics = ['log_slope_local', 'auc_log_k', 'auc_linear_k', 'early_ratio', 'mean_gain_128']

print(f"\n{'Metric':<20} {'ICC(task)':>12} {'%Task':>10} {'Interpretation':>20}")
print("-"*65)

for metric in metrics:
    result = compute_icc_oneway(df_with_task, metric, 'task')
    if result:
        task_icc_results[metric] = result
        icc = result['icc']
        
        if icc < 0.05:
            interp = "negligible"
        elif icc < 0.10:
            interp = "weak"
        elif icc < 0.20:
            interp = "moderate"
        else:
            interp = "substantial"
        
        print(f"{metric:<20} {icc:>12.3f} {result['pct_between']:>9.1f}% {interp:>20}")

In [ ]:
# Task ICC after controlling for length and fluency
print("\n" + "="*80)
print("TASK ICC AFTER CONTROLLING FOR LENGTH + FLUENCY")
print("="*80)

df_task_resid = df_with_task.copy()

for metric in metrics:
    valid_mask = (df_task_resid[metric].notna() & 
                  df_task_resid['token_count'].notna() & 
                  df_task_resid['baseline_nll'].notna())
    
    if valid_mask.sum() > 10:
        formula = f'{metric} ~ token_count + baseline_nll'
        model = smf.ols(formula, data=df_task_resid[valid_mask]).fit()
        
        resid_col = f'{metric}_resid'
        df_task_resid[resid_col] = np.nan
        df_task_resid.loc[valid_mask, resid_col] = model.resid

print(f"\n{'Metric':<20} {'ICC (raw)':>12} {'ICC (resid)':>12} {'Change':>10}")
print("-"*60)

task_icc_resid = {}
for metric in metrics:
    resid_col = f'{metric}_resid'
    if resid_col in df_task_resid.columns:
        result_resid = compute_icc_oneway(df_task_resid, resid_col, 'task')
        if result_resid and metric in task_icc_results:
            task_icc_resid[metric] = result_resid
            icc_raw = task_icc_results[metric]['icc']
            icc_resid = result_resid['icc']
            change = icc_resid - icc_raw
            print(f"{metric:<20} {icc_raw:>12.3f} {icc_resid:>12.3f} {change:>+10.3f}")

print("\nInterpretation:")
print("  If ICC(task) < 0.05: Safe to ignore task in longitudinal analysis")
print("  If ICC(task) > 0.10: Consider controlling for task")

## 6. Child-Level ICC Analysis

**Main Question**: Do children have stable "signatures" across sessions?

In [ ]:
print("="*80)
print("CHILD-LEVEL ICC: Is there a stable within-child signature?")
print("="*80)

# Filter to children with 2+ sessions
multi_session = df.groupby('subject_id').filter(lambda x: len(x) >= 2)
print(f"\nTranscripts from children with 2+ sessions: {len(multi_session)}")
print(f"Unique children: {multi_session['subject_id'].nunique()}")

child_icc_results = {}

print(f"\n{'Metric':<20} {'ICC(child)':>12} {'%Child':>10} {'Interpretation':>20}")
print("-"*65)

for metric in metrics:
    result = compute_icc_oneway(multi_session, metric, 'subject_id')
    if result:
        child_icc_results[metric] = result
        icc = result['icc']
        
        if icc < 0.10:
            interp = "weak - no signature"
        elif icc < 0.20:
            interp = "moderate"
        elif icc < 0.30:
            interp = "good - signature!"
        else:
            interp = "strong - clear sig!"
        
        print(f"{metric:<20} {icc:>12.3f} {result['pct_between']:>9.1f}% {interp:>20}")

In [ ]:
# Child ICC after controlling for task, length, and fluency
print("\n" + "="*80)
print("CHILD ICC AFTER CONTROLLING FOR TASK + LENGTH + FLUENCY")
print("="*80)

# Use subset with task identified
multi_with_task = multi_session[multi_session['task'].notna()].copy()
print(f"\nChildren with 2+ sessions AND task identified: {multi_with_task['subject_id'].nunique()}")

for metric in metrics:
    valid_mask = (multi_with_task[metric].notna() & 
                  multi_with_task['token_count'].notna() & 
                  multi_with_task['baseline_nll'].notna())
    
    if valid_mask.sum() > 10:
        formula = f'{metric} ~ C(task) + token_count + baseline_nll'
        try:
            model = smf.ols(formula, data=multi_with_task[valid_mask]).fit()
            resid_col = f'{metric}_resid_full'
            multi_with_task[resid_col] = np.nan
            multi_with_task.loc[valid_mask, resid_col] = model.resid
        except:
            pass

print(f"\n{'Metric':<20} {'ICC (raw)':>12} {'ICC (resid)':>12} {'Change':>10}")
print("-"*60)

child_icc_resid = {}
for metric in metrics:
    resid_col = f'{metric}_resid_full'
    if resid_col in multi_with_task.columns:
        result_resid = compute_icc_oneway(multi_with_task, resid_col, 'subject_id')
        if result_resid and metric in child_icc_results:
            child_icc_resid[metric] = result_resid
            icc_raw = child_icc_results[metric]['icc']
            icc_resid = result_resid['icc']
            change = icc_resid - icc_raw
            print(f"{metric:<20} {icc_raw:>12.3f} {icc_resid:>12.3f} {change:>+10.3f}")

print("\nInterpretation:")
print("  If ICC(child) ~ ICC(raw): Signature is real, not confounded")
print("  If ICC(child) << ICC(raw): Signature was partly task/length/fluency")

## 7. Matched-Pair Distance Analysis

**Goal**: Show that two sessions from the same child are more similar than sessions from different children, controlling for confounds.

**Design**:
1. **Within-child pairs**: All pairs of transcripts from the same child (YR1-YR2, YR2-YR3, YR1-YR3)
2. **Between-child pairs**: Matched on age (±3 months), task (if possible), token_count (±20%)
3. **Residualized version**: Distances on metrics after regressing out age, task, token_count, baseline_nll

In [ ]:
# Create within-child pairs
print("="*80)
print("CREATING SESSION PAIRS")
print("="*80)

within_pairs = []
multi_session_df = df.groupby('subject_id').filter(lambda x: len(x) >= 2)

for child_id, group in multi_session_df.groupby('subject_id'):
    group = group.sort_values('study_year')
    rows = group.to_dict('records')

    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            within_pairs.append({
                'child_id': child_id,
                'pair_type': 'within',
                'doc_id_1': rows[i]['doc_id'],
                'doc_id_2': rows[j]['doc_id'],
                'age_1': rows[i]['age_months'],
                'age_2': rows[j]['age_months'],
                'task_1': rows[i]['task'],
                'task_2': rows[j]['task'],
                'token_count_1': rows[i]['token_count'],
                'token_count_2': rows[j]['token_count'],
                'yr_1': rows[i]['study_year'],
                'yr_2': rows[j]['study_year'],
            })

print(f"Within-child pairs: {len(within_pairs)}")

# Year combinations
yr_combos = {}
for p in within_pairs:
    combo = f"YR{p['yr_1']}-YR{p['yr_2']}"
    yr_combos[combo] = yr_combos.get(combo, 0) + 1
print(f"Year combinations: {yr_combos}")

In [ ]:
# Create matched between-child pairs
print("\n" + "="*80)
print("CREATING MATCHED BETWEEN-CHILD PAIRS")
print("="*80)

# Matching tolerances
AGE_TOLERANCE = 3  # months
TOKEN_TOLERANCE = 0.20  # 20%

def find_matched_between_pair(target_row, df, rng, match_task=True):
    """Find a matched between-child pair for a target transcript."""
    candidates = df[df['subject_id'] != target_row['subject_id']].copy()

    # Age match (±3 months)
    if target_row['age_months'] is not None:
        candidates = candidates[
            (candidates['age_months'] >= target_row['age_months'] - AGE_TOLERANCE) &
            (candidates['age_months'] <= target_row['age_months'] + AGE_TOLERANCE)
        ]

    # Task match (if requested and available)
    if match_task and target_row['task'] is not None:
        task_matched = candidates[candidates['task'] == target_row['task']]
        if len(task_matched) >= 1:
            candidates = task_matched

    # Token count match (±20%)
    min_tokens = target_row['token_count'] * (1 - TOKEN_TOLERANCE)
    max_tokens = target_row['token_count'] * (1 + TOKEN_TOLERANCE)
    token_matched = candidates[
        (candidates['token_count'] >= min_tokens) &
        (candidates['token_count'] <= max_tokens)
    ]
    if len(token_matched) >= 1:
        candidates = token_matched

    if len(candidates) == 0:
        return None

    # Random selection from candidates
    return candidates.sample(1, random_state=rng).iloc[0]


# Create between-child pairs by matching each within-pair
rng_match = np.random.default_rng(RANDOM_SEED)
between_pairs = []
match_stats = {'total': 0, 'age_matched': 0, 'task_matched': 0, 'token_matched': 0}

for wp in within_pairs:
    # Get the two transcripts
    row1 = df[df['doc_id'] == wp['doc_id_1']].iloc[0]
    row2 = df[df['doc_id'] == wp['doc_id_2']].iloc[0]

    # Find matched partners for each
    match1 = find_matched_between_pair(row1, df, rng_match)
    match2 = find_matched_between_pair(row2, df, rng_match)

    if match1 is not None and match2 is not None and match1['subject_id'] != match2['subject_id']:
        between_pairs.append({
            'child_id': f"{match1['subject_id']}_vs_{match2['subject_id']}",
            'pair_type': 'between',
            'doc_id_1': match1['doc_id'],
            'doc_id_2': match2['doc_id'],
            'age_1': match1['age_months'],
            'age_2': match2['age_months'],
            'task_1': match1['task'],
            'task_2': match2['task'],
            'token_count_1': match1['token_count'],
            'token_count_2': match2['token_count'],
            'yr_1': match1['study_year'],
            'yr_2': match2['study_year'],
        })

        # Track matching quality
        match_stats['total'] += 1
        if abs(row1['age_months'] - match1['age_months']) <= AGE_TOLERANCE:
            match_stats['age_matched'] += 1
        if row1['task'] == match1['task']:
            match_stats['task_matched'] += 1

print(f"Between-child pairs created: {len(between_pairs)}")
print(f"Matching quality:")
print(f"  Age-matched: {match_stats['age_matched']}/{match_stats['total']} ({100*match_stats['age_matched']/max(1,match_stats['total']):.1f}%)")
print(f"  Task-matched: {match_stats['task_matched']}/{match_stats['total']} ({100*match_stats['task_matched']/max(1,match_stats['total']):.1f}%)")

In [ ]:
# Compute distances for all pairs
print("\n" + "="*80)
print("COMPUTING PAIRWISE DISTANCES")
print("="*80)

# Create lookup for metrics
metric_lookup = df.set_index('doc_id')[metrics].to_dict('index')

def compute_pair_distances(pairs, metric_lookup, metrics):
    """Compute absolute metric differences for all pairs."""
    results = []
    for p in pairs:
        row = {'pair_type': p['pair_type'], 'child_id': p['child_id']}

        m1 = metric_lookup.get(p['doc_id_1'], {})
        m2 = metric_lookup.get(p['doc_id_2'], {})

        for metric in metrics:
            v1, v2 = m1.get(metric), m2.get(metric)
            if v1 is not None and v2 is not None and not np.isnan(v1) and not np.isnan(v2):
                row[f'{metric}_diff'] = abs(v1 - v2)
            else:
                row[f'{metric}_diff'] = np.nan

        results.append(row)

    return pd.DataFrame(results)

all_pairs = within_pairs + between_pairs
df_pairs = compute_pair_distances(all_pairs, metric_lookup, metrics)

print(f"Total pairs: {len(df_pairs)}")
print(f"  Within-child: {len(df_pairs[df_pairs['pair_type'] == 'within'])}")
print(f"  Between-child: {len(df_pairs[df_pairs['pair_type'] == 'between'])}")

In [ ]:
# RAW distance analysis
print("="*80)
print("RAW DISTANCE ANALYSIS (no residualization)")
print("="*80)

distance_results_raw = {}

print(f"\n{'Metric':<20} {'Within':>12} {'Between':>12} {'Ratio':>10} {'p-value':>12}")
print("-"*70)

for metric in metrics:
    diff_col = f'{metric}_diff'

    within = df_pairs[df_pairs['pair_type'] == 'within'][diff_col].dropna()
    between = df_pairs[df_pairs['pair_type'] == 'between'][diff_col].dropna()

    if len(within) > 5 and len(between) > 5:
        stat, pval = stats.mannwhitneyu(within, between, alternative='less')
        ratio = within.mean() / between.mean() if between.mean() > 0 else np.nan

        distance_results_raw[metric] = {
            'within': within.values,
            'between': between.values,
            'within_mean': within.mean(),
            'between_mean': between.mean(),
            'ratio': ratio,
            'pval': pval,
        }

        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"{metric:<20} {within.mean():>12.4f} {between.mean():>12.4f} {ratio:>10.3f} {pval:>11.4f}{sig}")

print("\n* p<0.05, ** p<0.01, *** p<0.001 (one-tailed: within < between)")
print("Ratio < 1 = signature exists (within-child more similar)")

In [ ]:
# RESIDUALIZED distance analysis
print("="*80)
print("RESIDUALIZED DISTANCE ANALYSIS")
print("="*80)
print("Residualizing metrics on: age_months + C(task) + token_count + baseline_nll")

# Compute residuals on full dataset
df_resid = df.copy()

for metric in metrics:
    valid_mask = (df_resid[metric].notna() &
                  df_resid['age_months'].notna() &
                  df_resid['token_count'].notna() &
                  df_resid['baseline_nll'].notna())

    # Try with task, fall back without
    try:
        task_mask = valid_mask & df_resid['task'].notna()
        if task_mask.sum() > 20:
            formula = f'{metric} ~ age_months + C(task) + token_count + baseline_nll'
            model = smf.ols(formula, data=df_resid[task_mask]).fit()
            df_resid.loc[task_mask, f'{metric}_resid'] = model.resid

            # For rows without task, use simpler model
            no_task_mask = valid_mask & df_resid['task'].isna()
            if no_task_mask.sum() > 0:
                formula2 = f'{metric} ~ age_months + token_count + baseline_nll'
                model2 = smf.ols(formula2, data=df_resid[no_task_mask]).fit()
                df_resid.loc[no_task_mask, f'{metric}_resid'] = model2.resid
        else:
            formula = f'{metric} ~ age_months + token_count + baseline_nll'
            model = smf.ols(formula, data=df_resid[valid_mask]).fit()
            df_resid.loc[valid_mask, f'{metric}_resid'] = model.resid
    except Exception as e:
        print(f"  Warning: Could not residualize {metric}: {e}")

# Compute residualized distances
resid_metrics = [f'{m}_resid' for m in metrics if f'{m}_resid' in df_resid.columns]
metric_lookup_resid = df_resid.set_index('doc_id')[resid_metrics].to_dict('index')

# Map back to original metric names
def compute_resid_pair_distances(pairs, lookup, metrics):
    results = []
    for p in pairs:
        row = {'pair_type': p['pair_type'], 'child_id': p['child_id']}

        m1 = lookup.get(p['doc_id_1'], {})
        m2 = lookup.get(p['doc_id_2'], {})

        for metric in metrics:
            resid_col = f'{metric}_resid'
            v1, v2 = m1.get(resid_col), m2.get(resid_col)
            if v1 is not None and v2 is not None and not np.isnan(v1) and not np.isnan(v2):
                row[f'{metric}_resid_diff'] = abs(v1 - v2)
            else:
                row[f'{metric}_resid_diff'] = np.nan

        results.append(row)

    return pd.DataFrame(results)

df_pairs_resid = compute_resid_pair_distances(all_pairs, metric_lookup_resid, metrics)

# Analyze residualized distances
distance_results_resid = {}

print(f"\n{'Metric':<20} {'Within':>12} {'Between':>12} {'Ratio':>10} {'p-value':>12}")
print("-"*70)

for metric in metrics:
    diff_col = f'{metric}_resid_diff'

    if diff_col not in df_pairs_resid.columns:
        continue

    within = df_pairs_resid[df_pairs_resid['pair_type'] == 'within'][diff_col].dropna()
    between = df_pairs_resid[df_pairs_resid['pair_type'] == 'between'][diff_col].dropna()

    if len(within) > 5 and len(between) > 5:
        stat, pval = stats.mannwhitneyu(within, between, alternative='less')
        ratio = within.mean() / between.mean() if between.mean() > 0 else np.nan

        distance_results_resid[metric] = {
            'within': within.values,
            'between': between.values,
            'within_mean': within.mean(),
            'between_mean': between.mean(),
            'ratio': ratio,
            'pval': pval,
        }

        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"{metric:<20} {within.mean():>12.4f} {between.mean():>12.4f} {ratio:>10.3f} {pval:>11.4f}{sig}")

print("\nResidual ratio < 1 = signature survives after controlling for confounds")

In [ ]:
# Bootstrap permutation test for more robust inference
print("="*80)
print("BOOTSTRAP PERMUTATION TEST (1000 iterations)")
print("="*80)

N_PERMUTATIONS = 1000
rng_boot = np.random.default_rng(RANDOM_SEED)

bootstrap_results = {}

for metric in ['log_slope_local', 'mean_gain_128']:
    print(f"\n{metric}:")
    
    # Raw
    diff_col = f'{metric}_diff'
    valid_pairs = df_pairs[df_pairs[diff_col].notna()].copy()
    
    within_raw = valid_pairs[valid_pairs['pair_type'] == 'within'][diff_col].values
    between_raw = valid_pairs[valid_pairs['pair_type'] == 'between'][diff_col].values
    
    observed_diff = within_raw.mean() - between_raw.mean()
    
    # Permutation test
    all_values = np.concatenate([within_raw, between_raw])
    n_within = len(within_raw)
    
    perm_diffs = []
    for _ in range(N_PERMUTATIONS):
        perm_idx = rng_boot.permutation(len(all_values))
        perm_within = all_values[perm_idx[:n_within]]
        perm_between = all_values[perm_idx[n_within:]]
        perm_diffs.append(perm_within.mean() - perm_between.mean())
    
    pval_boot_raw = (np.sum(np.array(perm_diffs) <= observed_diff) + 1) / (N_PERMUTATIONS + 1)
    
    # Residualized
    resid_diff_col = f'{metric}_resid_diff'
    if resid_diff_col in df_pairs_resid.columns:
        valid_pairs_resid = df_pairs_resid[df_pairs_resid[resid_diff_col].notna()].copy()
        
        within_resid = valid_pairs_resid[valid_pairs_resid['pair_type'] == 'within'][resid_diff_col].values
        between_resid = valid_pairs_resid[valid_pairs_resid['pair_type'] == 'between'][resid_diff_col].values
        
        observed_diff_resid = within_resid.mean() - between_resid.mean()
        
        all_values_resid = np.concatenate([within_resid, between_resid])
        n_within_resid = len(within_resid)
        
        perm_diffs_resid = []
        for _ in range(N_PERMUTATIONS):
            perm_idx = rng_boot.permutation(len(all_values_resid))
            perm_within = all_values_resid[perm_idx[:n_within_resid]]
            perm_between = all_values_resid[perm_idx[n_within_resid:]]
            perm_diffs_resid.append(perm_within.mean() - perm_between.mean())
        
        pval_boot_resid = (np.sum(np.array(perm_diffs_resid) <= observed_diff_resid) + 1) / (N_PERMUTATIONS + 1)
    else:
        pval_boot_resid = np.nan
    
    bootstrap_results[metric] = {
        'pval_raw': pval_boot_raw,
        'pval_resid': pval_boot_resid,
    }
    
    print(f"  Raw: bootstrap p = {pval_boot_raw:.4f}")
    print(f"  Residualized: bootstrap p = {pval_boot_resid:.4f}")

## 8. Summary Table

In [ ]:
# MASTER SUMMARY TABLE
print("="*100)
print("MASTER SUMMARY TABLE: Within-Child Signature Analysis")
print("="*100)

summary_rows = []

for metric in ['log_slope_local', 'mean_gain_128']:
    row = {'Metric': metric}
    
    # Child ICC raw
    if metric in child_icc_results:
        row['Child ICC (raw)'] = f"{child_icc_results[metric]['icc']:.3f}"
    else:
        row['Child ICC (raw)'] = 'N/A'
    
    # Child ICC controlled (from earlier analysis)
    if metric in child_icc_resid:
        row['Child ICC (ctrl)'] = f"{child_icc_resid[metric]['icc']:.3f}"
    else:
        row['Child ICC (ctrl)'] = 'N/A'
    
    # Task ICC controlled
    if metric in task_icc_resid:
        row['Task ICC (ctrl)'] = f"{task_icc_resid[metric]['icc']:.3f}"
    else:
        row['Task ICC (ctrl)'] = 'N/A'
    
    # Within/between ratio (raw)
    if metric in distance_results_raw:
        row['W/B Ratio (raw)'] = f"{distance_results_raw[metric]['ratio']:.3f}"
        row['p-val (raw)'] = f"{distance_results_raw[metric]['pval']:.4f}"
    else:
        row['W/B Ratio (raw)'] = 'N/A'
        row['p-val (raw)'] = 'N/A'
    
    # Within/between ratio (residualized)
    if metric in distance_results_resid:
        row['W/B Ratio (resid)'] = f"{distance_results_resid[metric]['ratio']:.3f}"
        row['p-val (resid)'] = f"{distance_results_resid[metric]['pval']:.4f}"
    else:
        row['W/B Ratio (resid)'] = 'N/A'
        row['p-val (resid)'] = 'N/A'
    
    # Bootstrap p-values
    if metric in bootstrap_results:
        row['Bootstrap p (raw)'] = f"{bootstrap_results[metric]['pval_raw']:.4f}"
        row['Bootstrap p (resid)'] = f"{bootstrap_results[metric]['pval_resid']:.4f}"
    else:
        row['Bootstrap p (raw)'] = 'N/A'
        row['Bootstrap p (resid)'] = 'N/A'
    
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\n")
print(df_summary.to_string(index=False))

# Interpretation
print("\n" + "="*100)
print("INTERPRETATION GUIDE")
print("="*100)
print("""
- Child ICC >= 0.20 after controls = REAL SIGNATURE
- Task ICC < 0.10 after controls = Task is not a major confound
- W/B Ratio < 1.0 with p < 0.05 = Within-child pairs are more similar
- If both Child ICC and W/B Ratio survive residualization = STRONG EVIDENCE

VERDICT for log_slope_local:
""")

if 'log_slope_local' in child_icc_resid and 'log_slope_local' in distance_results_resid:
    child_icc = child_icc_resid['log_slope_local']['icc']
    wb_ratio = distance_results_resid['log_slope_local']['ratio']
    wb_pval = distance_results_resid['log_slope_local']['pval']
    
    if child_icc >= 0.20 and wb_ratio < 1.0 and wb_pval < 0.05:
        print(f"  STRONG SIGNATURE: Child ICC = {child_icc:.3f}, W/B ratio = {wb_ratio:.3f}, p = {wb_pval:.4f}")
    elif child_icc >= 0.15 or (wb_ratio < 0.9 and wb_pval < 0.05):
        print(f"  MODERATE SIGNATURE: Child ICC = {child_icc:.3f}, W/B ratio = {wb_ratio:.3f}, p = {wb_pval:.4f}")
    else:
        print(f"  WEAK/NO SIGNATURE: Child ICC = {child_icc:.3f}, W/B ratio = {wb_ratio:.3f}, p = {wb_pval:.4f}")
else:
    print("  Insufficient data to evaluate")

## 9. Key Figures

In [ ]:
# FIGURE 1: ICC Comparison (Child vs Task, Raw vs Controlled)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

key_metrics = ['log_slope_local', 'mean_gain_128']
x = np.arange(len(key_metrics))
width = 0.35

# Left panel: Child ICC
ax = axes[0]
raw_vals = [child_icc_results.get(m, {}).get('icc', np.nan) for m in key_metrics]
ctrl_vals = [child_icc_resid.get(m, {}).get('icc', np.nan) for m in key_metrics]

bars1 = ax.bar(x - width/2, raw_vals, width, label='Raw', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, ctrl_vals, width, label='Controlled', color='#2ecc71', alpha=0.8)

ax.set_ylabel('ICC', fontsize=12)
ax.set_title('Child ICC\\n(higher = stronger signature)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(key_metrics, fontsize=10)
ax.axhline(0.20, color='red', linestyle='--', alpha=0.5, label='Good threshold (0.20)')
ax.axhline(0.15, color='orange', linestyle=':', alpha=0.5, label='Moderate threshold (0.15)')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, max(0.5, max([v for v in raw_vals + ctrl_vals if not np.isnan(v)]) * 1.2))
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax.annotate(f'{height:.2f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

# Right panel: Task ICC
ax = axes[1]
task_raw = [task_icc_results.get(m, {}).get('icc', np.nan) for m in key_metrics]
task_ctrl = [task_icc_resid.get(m, {}).get('icc', np.nan) for m in key_metrics]

bars1 = ax.bar(x - width/2, task_raw, width, label='Raw', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, task_ctrl, width, label='Controlled', color='#f39c12', alpha=0.8)

ax.set_ylabel('ICC', fontsize=12)
ax.set_title('Task ICC\\n(lower = task not confounding)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(key_metrics, fontsize=10)
ax.axhline(0.10, color='green', linestyle='--', alpha=0.5, label='Negligible threshold (0.10)')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, max(0.3, max([v for v in task_raw + task_ctrl if not np.isnan(v)]) * 1.2))
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax.annotate(f'{height:.2f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

plt.suptitle('ICC Comparison: Child vs Task Effects', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'figure1_icc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 1 saved: figure1_icc_comparison.png")

In [ ]:
# FIGURE 2: Within vs Between Distance Overlay (Raw + Residualized)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metric = 'log_slope_local'

# Top row: Raw
if metric in distance_results_raw:
    ax = axes[0, 0]
    result = distance_results_raw[metric]
    
    ax.hist(result['within'], bins=25, alpha=0.6, density=True,
            label=f'Within-child (n={len(result["within"])}, mean={result["within_mean"]:.4f})',
            color='#2ecc71', edgecolor='#27ae60')
    ax.hist(result['between'], bins=25, alpha=0.6, density=True,
            label=f'Between-child (n={len(result["between"])}, mean={result["between_mean"]:.4f})',
            color='#e74c3c', edgecolor='#c0392b')
    
    ax.axvline(result['within_mean'], color='#27ae60', linestyle='--', linewidth=2)
    ax.axvline(result['between_mean'], color='#c0392b', linestyle='--', linewidth=2)
    
    ax.set_xlabel(f'|{metric}(session1) - {metric}(session2)|', fontsize=10)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'log_slope_local (RAW)\\nRatio = {result["ratio"]:.3f}, p = {result["pval"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

# Top right: Raw for mean_gain_128
metric2 = 'mean_gain_128'
if metric2 in distance_results_raw:
    ax = axes[0, 1]
    result = distance_results_raw[metric2]
    
    ax.hist(result['within'], bins=25, alpha=0.6, density=True,
            label=f'Within-child (mean={result["within_mean"]:.4f})',
            color='#2ecc71', edgecolor='#27ae60')
    ax.hist(result['between'], bins=25, alpha=0.6, density=True,
            label=f'Between-child (mean={result["between_mean"]:.4f})',
            color='#e74c3c', edgecolor='#c0392b')
    
    ax.axvline(result['within_mean'], color='#27ae60', linestyle='--', linewidth=2)
    ax.axvline(result['between_mean'], color='#c0392b', linestyle='--', linewidth=2)
    
    ax.set_xlabel(f'|{metric2}(session1) - {metric2}(session2)|', fontsize=10)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'mean_gain_128 (RAW)\\nRatio = {result["ratio"]:.3f}, p = {result["pval"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

# Bottom row: Residualized
metric = 'log_slope_local'
if metric in distance_results_resid:
    ax = axes[1, 0]
    result = distance_results_resid[metric]
    
    ax.hist(result['within'], bins=25, alpha=0.6, density=True,
            label=f'Within-child (mean={result["within_mean"]:.4f})',
            color='#3498db', edgecolor='#2980b9')
    ax.hist(result['between'], bins=25, alpha=0.6, density=True,
            label=f'Between-child (mean={result["between_mean"]:.4f})',
            color='#9b59b6', edgecolor='#8e44ad')
    
    ax.axvline(result['within_mean'], color='#2980b9', linestyle='--', linewidth=2)
    ax.axvline(result['between_mean'], color='#8e44ad', linestyle='--', linewidth=2)
    
    ax.set_xlabel(f'|{metric}_resid(session1) - {metric}_resid(session2)|', fontsize=10)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'log_slope_local (RESIDUALIZED)\\nRatio = {result["ratio"]:.3f}, p = {result["pval"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

# Bottom right: Residualized for mean_gain_128
metric2 = 'mean_gain_128'
if metric2 in distance_results_resid:
    ax = axes[1, 1]
    result = distance_results_resid[metric2]
    
    ax.hist(result['within'], bins=25, alpha=0.6, density=True,
            label=f'Within-child (mean={result["within_mean"]:.4f})',
            color='#3498db', edgecolor='#2980b9')
    ax.hist(result['between'], bins=25, alpha=0.6, density=True,
            label=f'Between-child (mean={result["between_mean"]:.4f})',
            color='#9b59b6', edgecolor='#8e44ad')
    
    ax.axvline(result['within_mean'], color='#2980b9', linestyle='--', linewidth=2)
    ax.axvline(result['between_mean'], color='#8e44ad', linestyle='--', linewidth=2)
    
    ax.set_xlabel(f'|{metric2}_resid(session1) - {metric2}_resid(session2)|', fontsize=10)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'mean_gain_128 (RESIDUALIZED)\\nRatio = {result["ratio"]:.3f}, p = {result["pval"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Within vs Between Child Distances\\nTop: Raw | Bottom: Residualized (controlling for age, task, length, fluency)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'figure2_within_vs_between_distances.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure 2 saved: figure2_within_vs_between_distances.png")

## 10. Save Results

In [ ]:
# Save all results
print("="*80)
print("SAVING RESULTS")
print("="*80)

# 1. Transcript-level metrics
df.to_csv(output_dir / 'transcript_level_metrics.csv', index=False)
print(f"Saved: transcript_level_metrics.csv ({len(df)} rows)")

# 2. Pairwise distances (for reproducibility)
df_pairs.to_csv(output_dir / 'pairwise_distances_raw.csv', index=False)
print(f"Saved: pairwise_distances_raw.csv ({len(df_pairs)} rows)")

if len(df_pairs_resid) > 0:
    df_pairs_resid.to_csv(output_dir / 'pairwise_distances_residualized.csv', index=False)
    print(f"Saved: pairwise_distances_residualized.csv ({len(df_pairs_resid)} rows)")

# 3. Summary table
df_summary.to_csv(output_dir / 'summary_table.csv', index=False)
print(f"Saved: summary_table.csv")

# 4. Full text report
with open(output_dir / 'signature_analysis_report.txt', 'w') as f:
    f.write("ECSC WITHIN-CHILD SIGNATURE ANALYSIS - FULL REPORT\\n")
    f.write("="*80 + "\\n\\n")
    
    f.write("DATASET\\n")
    f.write("-"*40 + "\\n")
    f.write(f"Total transcripts analyzed: {len(df)}\\n")
    f.write(f"Unique children: {df['subject_id'].nunique()}\\n")
    f.write(f"Children with 2+ sessions: {multi_session['subject_id'].nunique()}\\n")
    f.write(f"Within-child pairs: {len(within_pairs)}\\n")
    f.write(f"Matched between-child pairs: {len(between_pairs)}\\n")
    f.write(f"Transcripts with task identified: {df['task'].notna().sum()}\\n\\n")
    
    f.write("MATCHING QUALITY\\n")
    f.write("-"*40 + "\\n")
    f.write(f"Age tolerance: ±{AGE_TOLERANCE} months\\n")
    f.write(f"Token count tolerance: ±{int(TOKEN_TOLERANCE*100)}%\\n")
    f.write(f"Age-matched pairs: {match_stats['age_matched']}/{match_stats['total']}\\n")
    f.write(f"Task-matched pairs: {match_stats['task_matched']}/{match_stats['total']}\\n\\n")
    
    f.write("TASK ICC (controlled for length + fluency)\\n")
    f.write("-"*40 + "\\n")
    for metric in metrics:
        if metric in task_icc_resid:
            f.write(f"{metric}: {task_icc_resid[metric]['icc']:.4f}\\n")
    f.write("\\n")
    
    f.write("CHILD ICC (controlled for task + length + fluency)\\n")
    f.write("-"*40 + "\\n")
    for metric in metrics:
        if metric in child_icc_resid:
            f.write(f"{metric}: {child_icc_resid[metric]['icc']:.4f}\\n")
    f.write("\\n")
    
    f.write("WITHIN vs BETWEEN DISTANCES (RAW)\\n")
    f.write("-"*40 + "\\n")
    for metric in metrics:
        if metric in distance_results_raw:
            r = distance_results_raw[metric]
            f.write(f"{metric}: within={r['within_mean']:.4f}, between={r['between_mean']:.4f}, ")
            f.write(f"ratio={r['ratio']:.4f}, p={r['pval']:.4f}\\n")
    f.write("\\n")
    
    f.write("WITHIN vs BETWEEN DISTANCES (RESIDUALIZED)\\n")
    f.write("-"*40 + "\\n")
    for metric in metrics:
        if metric in distance_results_resid:
            r = distance_results_resid[metric]
            f.write(f"{metric}: within={r['within_mean']:.4f}, between={r['between_mean']:.4f}, ")
            f.write(f"ratio={r['ratio']:.4f}, p={r['pval']:.4f}\\n")
    f.write("\\n")
    
    f.write("BOOTSTRAP PERMUTATION TESTS\\n")
    f.write("-"*40 + "\\n")
    for metric, res in bootstrap_results.items():
        f.write(f"{metric}: p_raw={res['pval_raw']:.4f}, p_resid={res['pval_resid']:.4f}\\n")
    f.write("\\n")
    
    f.write("INTERPRETATION\\n")
    f.write("-"*40 + "\\n")
    f.write("Child ICC >= 0.20 after controls = REAL SIGNATURE\\n")
    f.write("Task ICC < 0.10 after controls = Task not confounding\\n")
    f.write("W/B Ratio < 1.0 with p < 0.05 = Signature exists\\n")

print(f"Saved: signature_analysis_report.txt")

# Summary
print("\\n" + "="*80)
print(f"All outputs saved to: {output_dir}")
print("="*80)
print("Files:")
print("  - transcript_level_metrics.csv")
print("  - pairwise_distances_raw.csv")
print("  - pairwise_distances_residualized.csv")
print("  - summary_table.csv")
print("  - signature_analysis_report.txt")
print("  - figure1_icc_comparison.png")
print("  - figure2_within_vs_between_distances.png")